In [ ]:
pip install esm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 

In [ ]:
import pandas as pd
import torch
import os
import numpy as np
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

In [ ]:
columns = [
    "pdb_id",
    "receptor_chain",
    "resolution",
    "binding_site_code",
    "ligand_id",
    "ligand_chain",
    "ligand_serial_number",
    "binding_residues_pdb",
    "binding_residues_seq",
    "catalytic_residues_pdb",
    "catalytic_residues_seq",
    "ec_number",
    "go_terms",
    "binding_affinity_manual",
    "binding_affinity_moad",
    "binding_affinity_pdbbind",
    "binding_affinity_bindingdb",
    "uniprot_id",
    "pubmed_id",
    "ligand_residue_seq_number",
    "sequence"
]

df = pd.read_csv("BioLiP_nr.txt.gz", sep="\t", names=columns, header=None)

df = df[['pdb_id', 'receptor_chain', 'resolution', 'ligand_id', 'binding_residues_seq','sequence']]

print(df.shape)
print(df.ligand_id.value_counts())
df.head()


/tmp/ipykernel_513/973534019.py:25: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("BioLiP_nr.txt.gz", sep="\t", names=columns, header=None)


(86458, 6)
ligand_id
rna      9528
ZN       8255
CLA      7354
CA       6176
dna      4231
         ... 
4CH         1
YS4         1
RRY         1
A1D6H       1
G50         1
Name: count, Length: 6019, dtype: int64


,pdb_id,receptor_chain,resolution,ligand_id,binding_residues_seq,sequence
0,10mh,A,2.55,dna,I86 Q90 R209 G236 Q237 G256 G257,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...
1,10mh,A,2.55,dna,F79 C81 S87 E119 R165 R228 Q237 R240 I249 T250...,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...
2,10mh,A,2.55,SAH,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...
3,11as,A,2.50,ASN,D43 A71 Q113 Y215 S248 R252,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...
4,11ba,A,2.06,UPA,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,KESAAAKFERQHMDSGNSPSSSSNYCNLMMCCRKMTQGKCKPVNTF...


In [ ]:
df = df[df['binding_residues_seq'].notna()]
df = df[df['ligand_id'].notna()]

to_remove = {
    # Metal ions
    "ZN", "CA", "MG", "MN", "FE", "FE2", "CU", "CU1", "CO", "NI", "CD", "HG", "MN3",
    "CL", "AG", "LA", "YB", "RB", "PB", "SE", "AU", "BR", "IOD",
    # Nucleic acids
    "RNA", "DNA", "PEPTIDE",
    # Crystallization artifacts
    "SO4", "GOL", "EDO", "PO4", "ACT", "BME", "MPD", "FMT", "ACE", "DMS", "PGE",
    "EOH", "TRS", "IMD", "PGW", "C8E", "C2E",
    # Free amino acids
    "GLY", "ALA", "VAL", "LEU", "ILE", "PRO", "PHE", "TRP", "SER", "THR", "CYS",
    "MET", "ASN", "GLN", "ASP", "GLU", "LYS", "ARG", "HIS", "TYR"
}

df_cleaned = df[~df["ligand_id"].str.upper().isin(to_remove)]
df_cleaned.reset_index(drop= True, inplace=True)
df_cleaned

,pdb_id,receptor_chain,resolution,ligand_id,binding_residues_seq,sequence
0,10mh,A,2.55,SAH,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...
1,11ba,A,2.06,UPA,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,KESAAAKFERQHMDSGNSPSSSSNYCNLMMCCRKMTQGKCKPVNTF...
2,13pk,A,2.50,ADP,G213 A214 K219 L311 P336 G338 V339 E341 G371 D...,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...
3,155c,A,2.50,HEM,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,NEGDAAKGEKEFNKCKACHMIQAPDGTDIKGGKTGPNLYGVVGRKI...
4,19hc,A,1.80,HEM,S8 A10 M16 F35 H37 H40 I44 C47 C50 H51 P56 V57...,AALEPTDSGAPSAIVMFPVGEKPNPKGAAMKPVVFNHLIHEKKIAD...
...,...,...,...,...,...,...
43983,9vf7,A,2.40,FMN,V17 N50 C51 R52 L54 Q55 V64 I71 L81 N83 N93 L9...,RDEIFRIAVETILAGVVITDAQLPDYPIVYCNPGFVQLTGYPSEEV...
43984,9vid,A,2.22,HEM,Y42 F43 H45 H58 K61 R82 L83 L86 H87 L91 N97 F9...,VLSMEDKSNVKAIWGKASGHLEEYGAEALERMFCAYPQTKIYFPHF...
43985,9vid,B,2.22,HEM,K38 F42 H44 F45 H63 V67 S70 L88 H92 F96 V98 N1...,ASFDAHERKFIVDLWAKVDVAQCGADALSRMLIVYPWKRRYFEHFG...
43986,9vmx,D,3.20,L9Q,R874 L875 F1213,ELVKGVYAKYWIYVCAGMFIVVSFAGRLVVYKIVYMFLFLLCLTLF...


In [ ]:
df_cleaned['sequence_length (aa)'] = df_cleaned['sequence'].apply(lambda x: len(x))
df_cleaned = df_cleaned[df_cleaned['resolution'] <= 2.50]
df_cleaned = df_cleaned[(df_cleaned['sequence_length (aa)'] >= 150) & (df_cleaned['sequence_length (aa)'] <= 800)]
df_cleaned = df_cleaned.drop_duplicates(['pdb_id','ligand_id'])
df_cleaned.reset_index(drop=True, inplace=True)
df_cleaned



/tmp/ipykernel_513/3719118623.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['sequence_length (aa)'] = df_cleaned['sequence'].apply(lambda x: len(x))


,pdb_id,receptor_chain,resolution,ligand_id,binding_residues_seq,sequence,sequence_length (aa)
0,13pk,A,2.500,ADP,G213 A214 K219 L311 P336 G338 V339 E341 G371 D...,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...,415
1,19hc,A,1.800,HEM,S8 A10 M16 F35 H37 H40 I44 C47 C50 H51 P56 V57...,AALEPTDSGAPSAIVMFPVGEKPNPKGAAMKPVVFNHLIHEKKIAD...,292
2,1a05,A,2.000,IPM,R95 R105 R133 Y140 D246,MKKIAIFAGDGIGPEIVAAARQVLDAVDQAAHLGLRCTEGLVGGAA...,357
3,1a0g,B,2.000,PMP,R50 K145 E177 S180 S181 G203 I204 T205 S240 T241,GYTLWNDQIVKDEEVKIDKEDRGYQFGDGVYEVVKVYNGEMFTVNE...,282
4,1a26,A,2.250,CNA,H165 M229 K242 L323 L324 Y325 E327,KSKLAKPIQDLIKMIFDVESMKKAMVEFEIDLQKMPLGKLSKRQIQ...,351
...,...,...,...,...,...,...,...
14564,9us4,B,1.950,AC1,T238 H334,AWTTTDFPAFTEEGTGRFISQKVVEKGTRPLQLNFDQQCWQPSGGI...,652
14565,9us4,B,1.950,GLC,H439 W480,AWTTTDFPAFTEEGTGRFISQKVVEKGTRPLQLNFDQQCWQPSGGI...,652
14566,9v14,A,0.930,42B,D12 C13 Y14 S121 N122 K152 E180 S181 H184,MQLTDFKALTFDCYGTLIDWETGIVNALQPLAKRTGKTFTSDELLE...,240
14567,9v59,A,2.175,GLC,M171 A174 R178,GTGFPFDPHYVEVLGERMHYVDVGPRDGTPVLFLHGNPTSSYVWRN...,462


In [ ]:
rows = []

for index, row in df_cleaned.iterrows():
  binding_residues = set()

  for binding_residue in row['binding_residues_seq'].split():
    position = int(''.join(filter(str.isdigit, binding_residue)))
    binding_residues.add(position)

  for i, residue in enumerate(row['sequence']):
    rows.append({
        'pdb_id': row['pdb_id'],
        'receptor_chain': row['receptor_chain'],
        'ligand_id': row['ligand_id'],
        'position': i+1,
        'residue': residue,
        'isbinding': 1 if i+1 in binding_residues else 0,
        })

residue_df = pd.DataFrame(rows)

In [ ]:
residue_df

,pdb_id,receptor_chain,ligand_id,position,residue,isbinding
0,13pk,A,ADP,1,E,0
1,13pk,A,ADP,2,K,0
2,13pk,A,ADP,3,K,0
3,13pk,A,ADP,4,S,0
4,13pk,A,ADP,5,I,0
...,...,...,...,...,...,...
5057166,9v59,A,A1EQX,458,T,0
5057167,9v59,A,A1EQX,459,L,0
5057168,9v59,A,A1EQX,460,E,0
5057169,9v59,A,A1EQX,461,I,0


In [ ]:
client = ESMC.from_pretrained("esmc_300m").to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

data/weights/esmc_300m_2024_12_v0.pth:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

In [ ]:
os.makedirs("embeddings_proteins2", exist_ok=True)

for _, entry in df_cleaned.iterrows():
    pdb_id    = entry["pdb_id"]
    chain     = entry["receptor_chain"]
    ligand    = entry["ligand_id"]

    unique_id = f"{pdb_id}_{chain}_{ligand}"
    save_path = f"embeddings_proteins2/{unique_id}.npy"

    if os.path.exists(save_path):
        continue

    seq = entry["sequence"]
    protein = ESMProtein(sequence=seq)
    protein_tensor = client.encode(protein)

    with torch.no_grad():
        output = client.logits(
            protein_tensor,
            LogitsConfig(sequence=True, return_embeddings=True)
        )

    embeddings = output.embeddings.squeeze(0).half().cpu().numpy()[1:-1, :]
    np.save(save_path, embeddings)

    del output, embeddings
    torch.cuda.empty_cache()

print("✅ All embeddings saved to disk")

✅ All embeddings saved to disk


In [ ]:
EMBEDDINGS_DIR = "embeddings_proteins2"
OUTPUT_DIR = "embedding_chunks2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for _, entry in df_cleaned.iterrows():
    pdb_id    = entry["pdb_id"]
    chain     = entry["receptor_chain"]
    ligand    = entry["ligand_id"]
    unique_id = f"{pdb_id}_{chain}_{ligand}"

    chunk_path = f"{OUTPUT_DIR}/{unique_id}.parquet"
    if os.path.exists(chunk_path):
        continue

    embeddings = np.load(f"{EMBEDDINGS_DIR}/{unique_id}.npy")

    meta = pd.DataFrame({
        "pdb_id":         pdb_id,
        "receptor_chain": chain,
        "ligand_id":      ligand,
        "position":       np.arange(1, len(embeddings) + 1)
    })

    esm_cols = pd.DataFrame(embeddings, columns=[f"esm_{i}" for i in range(embeddings.shape[1])])

    chunk_df = pd.concat([meta, esm_cols], axis=1)
    chunk_df.to_parquet(chunk_path, index=False)

    del embeddings, chunk_df, meta, esm_cols

print("✅ All chunks saved")

✅ All chunks saved


In [ ]:
embedding_df = pd.read_parquet(OUTPUT_DIR)
print(embedding_df.shape)
embedding_df

In [ ]:
df_merge = residue_df.merge(
    embedding_df,
    on=["pdb_id", "receptor_chain", "ligand_id", "position"],
    how="inner"
)

In [ ]:
df_merge

In [ ]:
"""
df_merge.to_parquet(
    "model_dataset.parquet",
    engine="pyarrow",
    compression="zstd"
)
"""

In [ ]:
# ESM helper class encapsulating the steps in this notebook to produce `df_merge`
import os
import numpy np
import pandas as pd
import torch
from typing import Optional
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig


class ESM:
    """Encapsulate dataset loading, cleaning, embedding extraction (ESM), chunking and merging.

    Methods:
     - load_biolip: read raw BioLiP file
     - clean: filter and clean raw dataframe
     - make_residue_df: expand sequences into per-residue rows with isbinding label
     - init_client: load ESM model to device
     - compute_embeddings: encode sequences and save per-protein .npy embeddings
     - chunk_embeddings: turn saved .npy embeddings into per-protein parquet chunks with esm_ columns
     - load_embedding_df: read all chunk parquet files into a single dataframe
     - merge: merge residue dataframe with embedding dataframe -> df_merge
     - save: write final df_merge to parquet
     - run_all: convenience to run entire pipeline
    """

    def __init__(
        self,
        model_name: str = "esmc_300m",
        device: Optional[str] = None,
        embeddings_dir: str = "embeddings_proteins2",
        chunks_dir: str = "embedding_chunks2",
    ):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.embeddings_dir = embeddings_dir
        self.chunks_dir = chunks_dir
        self.client: Optional[ESMC] = None

    def load_biolip(self, path: str = "BioLiP_nr.txt.gz") -> pd.DataFrame:
        columns = [
            "pdb_id",
            "receptor_chain",
            "resolution",
            "binding_site_code",
            "ligand_id",
            "ligand_chain",
            "ligand_serial_number",
            "binding_residues_pdb",
            "binding_residues_seq",
            "catalytic_residues_pdb",
            "catalytic_residues_seq",
            "ec_number",
            "go_terms",
            "binding_affinity_manual",
            "binding_affinity_moad",
            "binding_affinity_pdbbind",
            "binding_affinity_bindingdb",
            "uniprot_id",
            "pubmed_id",
            "ligand_residue_seq_number",
            "sequence",
        ]

        df = pd.read_csv(path, sep="\t", names=columns, header=None)
        df = df[["pdb_id", "receptor_chain", "resolution", "ligand_id", "binding_residues_seq", "sequence"]]
        return df

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df[df["binding_residues_seq"].notna()]
        df = df[df["ligand_id"].notna()]

        to_remove = {
            # Metal ions
            "ZN",
            "CA",
            "MG",
            "MN",
            "FE",
            "FE2",
            "CU",
            "CU1",
            "CO",
            "NI",
            "CD",
            "HG",
            "MN3",
            "CL",
            "AG",
            "LA",
            "YB",
            "RB",
            "PB",
            "SE",
            "AU",
            "BR",
            "IOD",
            # Nucleic acids
            "RNA",
            "DNA",
            "PEPTIDE",
            # Crystallization artifacts
            "SO4",
            "GOL",
            "EDO",
            "PO4",
            "ACT",
            "BME",
            "MPD",
            "FMT",
            "ACE",
            "DMS",
            "PGE",
            "EOH",
            "TRS",
            "IMD",
            "PGW",
            "C8E",
            "C2E",
            # Free amino acids
            "GLY",
            "ALA",
            "VAL",
            "LEU",
            "ILE",
            "PRO",
            "PHE",
            "TRP",
            "SER",
            "THR",
            "CYS",
            "MET",
            "ASN",
            "GLN",
            "ASP",
            "GLU",
            "LYS",
            "ARG",
            "HIS",
            "TYR",
        }

        df_cleaned = df[~df["ligand_id"].str.upper().isin(to_remove)].copy()
        df_cleaned.reset_index(drop=True, inplace=True)

        # compute sequence length and filter by resolution and length
        df_cleaned["sequence_length (aa)"] = df_cleaned["sequence"].apply(lambda x: len(x))
        df_cleaned = df_cleaned[df_cleaned["resolution"] <= 2.50]
        df_cleaned = df_cleaned[(df_cleaned["sequence_length (aa)"] >= 150) & (df_cleaned["sequence_length (aa)"] <= 800)]
        df_cleaned = df_cleaned.drop_duplicates(["pdb_id", "ligand_id"])
        df_cleaned.reset_index(drop=True, inplace=True)

        return df_cleaned

    def make_residue_df(self, df_cleaned: pd.DataFrame) -> pd.DataFrame:
        rows = []
        for _, row in df_cleaned.iterrows():
            binding_residues = set()
            for binding_residue in row["binding_residues_seq"].split():
                position = int("".join(filter(str.isdigit, binding_residue)))
                binding_residues.add(position)

            for i, residue in enumerate(row["sequence"]):
                rows.append(
                    {
                        "pdb_id": row["pdb_id"],
                        "receptor_chain": row["receptor_chain"],
                        "ligand_id": row["ligand_id"],
                        "position": i + 1,
                        "residue": residue,
                        "isbinding": 1 if i + 1 in binding_residues else 0,
                    }
                )

        residue_df = pd.DataFrame(rows)
        return residue_df

    def init_client(self):
        if self.client is None:
            self.client = ESMC.from_pretrained(self.model_name).to(self.device)

    def compute_embeddings(self, df_cleaned: pd.DataFrame, force: bool = False):
        os.makedirs(self.embeddings_dir, exist_ok=True)
        self.init_client()

        for _, entry in df_cleaned.iterrows():
            pdb_id = entry["pdb_id"]
            chain = entry["receptor_chain"]
            ligand = entry["ligand_id"]

            unique_id = f"{pdb_id}_{chain}_{ligand}"
            save_path = os.path.join(self.embeddings_dir, f"{unique_id}.npy")

            if os.path.exists(save_path) and not force:
                continue

            seq = entry["sequence"]
            protein = ESMProtein(sequence=seq)
            protein_tensor = self.client.encode(protein)

            with torch.no_grad():
                output = self.client.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))

            embeddings = output.embeddings.squeeze(0).half().cpu().numpy()[1:-1, :]
            np.save(save_path, embeddings)

            del output, embeddings
            torch.cuda.empty_cache()

        print("✅ All embeddings saved to disk")

    def chunk_embeddings(self, df_cleaned: pd.DataFrame, force: bool = False):
        os.makedirs(self.chunks_dir, exist_ok=True)

        for _, entry in df_cleaned.iterrows():
            pdb_id = entry["pdb_id"]
            chain = entry["receptor_chain"]
            ligand = entry["ligand_id"]
            unique_id = f"{pdb_id}_{chain}_{ligand}"

            chunk_path = os.path.join(self.chunks_dir, f"{unique_id}.parquet")
            if os.path.exists(chunk_path) and not force:
                continue

            embeddings_path = os.path.join(self.embeddings_dir, f"{unique_id}.npy")
            if not os.path.exists(embeddings_path):
                # skip if embedding missing
                continue

            embeddings = np.load(embeddings_path)

            meta = pd.DataFrame(
                {
                    "pdb_id": pdb_id,
                    "receptor_chain": chain,
                    "ligand_id": ligand,
                    "position": np.arange(1, len(embeddings) + 1),
                }
            )

            esm_cols = pd.DataFrame(embeddings, columns=[f"esm_{i}" for i in range(embeddings.shape[1])])

            chunk_df = pd.concat([meta, esm_cols], axis=1)
            chunk_df.to_parquet(chunk_path, index=False)

            del embeddings, chunk_df, meta, esm_cols

        print("✅ All chunks saved")

    def load_embedding_df(self) -> pd.DataFrame:
        # pandas can read a folder of parquet files (pyarrow engine) as a dataset
        df = pd.read_parquet(self.chunks_dir)
        return df

    def merge(self, residue_df: pd.DataFrame, embedding_df: pd.DataFrame) -> pd.DataFrame:
        df_merge = residue_df.merge(
            embedding_df, on=["pdb_id", "receptor_chain", "ligand_id", "position"], how="inner"
        )
        return df_merge

    def save(self, df_merge: pd.DataFrame, path: str = "model_dataset.parquet"):
        df_merge.to_parquet(path, engine="pyarrow", compression="zstd")

    def run_all(self, raw_path: str = "BioLiP_nr.txt.gz", save_path: str = "model_dataset.parquet", force: bool = False):
        df = self.load_biolip(raw_path)
        df_cleaned = self.clean(df)
        residue_df = self.make_residue_df(df_cleaned)
        self.compute_embeddings(df_cleaned, force=force)
        self.chunk_embeddings(df_cleaned, force=force)
        embedding_df = self.load_embedding_df()
        df_merge = self.merge(residue_df, embedding_df)
        self.save(df_merge, save_path)
        return df_merge


# Example usage (Cell 1):
# Instantiate and run the full pipeline (this will compute embeddings on GPU if available)
# esm = ESM()
# df_merge = esm.run_all(raw_path="BioLiP_nr.txt.gz", save_path="model_dataset.parquet")
